# Lab 7 — Variational Autoencoders and Controlled Latent Generation

This notebook contains two compact parts:

- **Part A:** a minimal **vanilla VAE** on **MNIST**, loaded via the `datasets` interface and implemented with **`ModelMixin` + `ConfigMixin`** for standardized save/load.
- **Part B:** a **pretrained T2I-Adapter** for **line-art colorization**, loaded in the `diffusers` style and used as a gentle control module on top of a frozen text-to-image diffusion backbone.

We assume the class has already seen the basic VAE theory. Accordingly, we keep only the **core equations**, omit historical detours, and avoid any Kaggle-flavored ceremony. Civilization survives.


## 0. Setup

We assume:

- PyTorch is available.
- `datasets`, `diffusers`, `Pillow`, and `matplotlib` are available.
- For Part B, `data/train.parquet` and `data/test.parquet` follow the same format as in **Lab 4**.

The notebook is written for a single GPU, but CPU execution is also acceptable for debugging and for those who enjoy watching epochs mature slowly, like cheese.


In [ ]:
import os
from pathlib import Path
import math
import random

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from datasets import load_dataset
from diffusers import ModelMixin, ConfigMixin, AutoencoderKL
from diffusers.configuration_utils import register_to_config

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

---

# Part A — Vanilla VAE on MNIST

## 1. Core equations

We assume the latent-variable model

$$
p_{\theta}(x, z) = p(z)\,p_{\theta}(x\mid z),
\qquad
p(z)=\mathcal{N}(0,I).
$$

We approximate the posterior by

$$
q_{\phi}(z\mid x)=\mathcal{N}\!\bigl(\mu_{\phi}(x), \operatorname{diag}(\sigma^2_{\phi}(x))\bigr).
$$

The evidence lower bound (ELBO) is

$$
\log p_{\theta}(x)
\ge
\mathcal{L}_{\mathrm{ELBO}}(x;\theta,\phi)
=
\mathbb{E}_{q_{\phi}(z\mid x)}\!\left[\log p_{\theta}(x\mid z)\right]
-
D_{\mathrm{KL}}\!\left(q_{\phi}(z\mid x)\,\|\,p(z)\right).
$$

Using the reparameterization trick,

$$
z = \mu_{\phi}(x) + \sigma_{\phi}(x)\odot \varepsilon,
\qquad
\varepsilon \sim \mathcal{N}(0,I).
$$

For a diagonal Gaussian posterior against the standard normal prior,

$$
D_{\mathrm{KL}}\!\left(q_{\phi}(z\mid x)\,\|\,p(z)\right)
=
\frac{1}{2}\sum_{j=1}^{d}
\left(
\mu_j^2 + \sigma_j^2 - \log \sigma_j^2 - 1
\right).
$$


## 2. Load MNIST with `datasets`

We deliberately use the `datasets` interface rather than `torchvision.datasets`, so that Part A and Part B follow a similar data-loading style.


In [ ]:
mnist = load_dataset("ylecun/mnist", "mnist")
mnist

In [ ]:
def pil_to_tensor32(img: Image.Image) -> torch.Tensor:
    img = img.convert("L")
    arr = np.array(img).astype(np.float32) / 255.0  # (28, 28)
    arr = np.pad(arr, ((2, 2), (2, 2)), mode="constant")  # -> (32, 32)
    return torch.from_numpy(arr)[None, ...]  # (1, 32, 32)


def mnist_collate_fn(batch):
    x = torch.stack([pil_to_tensor32(b["image"]) for b in batch], dim=0)
    y = torch.tensor([b["label"] for b in batch], dtype=torch.long)
    return {"image": x, "label": y}


batch_size = 128
train_loader = DataLoader(
    mnist["train"], batch_size=batch_size, shuffle=True, collate_fn=mnist_collate_fn
)
test_loader = DataLoader(
    mnist["test"], batch_size=batch_size, shuffle=False, collate_fn=mnist_collate_fn
)

batch = next(iter(train_loader))
print(batch["image"].shape, batch["label"].shape)

In [ ]:
def show_mnist_batch(x: torch.Tensor, y: torch.Tensor, n: int = 8):
    n = min(n, x.shape[0])
    plt.figure(figsize=(1.5 * n, 2))
    for i in range(n):
        plt.subplot(1, n, i + 1)
        plt.imshow(x[i, 0].cpu(), cmap="gray")
        plt.title(str(int(y[i])))
        plt.axis("off")
    plt.tight_layout()
    plt.show()


show_mnist_batch(batch["image"], batch["label"], n=8)

## 3. A minimal convolutional VAE with `ModelMixin` + `ConfigMixin`

The model below is intentionally small. It is a teaching model, not an ego project.


In [ ]:
class ToyVAE(ModelMixin, ConfigMixin):
    @register_to_config
    def __init__(self, latent_dim: int = 16, hidden_dims=(32, 64, 128)):
        super().__init__()
        h1, h2, h3 = hidden_dims

        self.encoder = nn.Sequential(
            nn.Conv2d(1, h1, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(h1, h2, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(h2, h3, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
        )

        self.fc_mu = nn.Linear(h3 * 4 * 4, latent_dim)
        self.fc_logvar = nn.Linear(h3 * 4 * 4, latent_dim)

        self.fc_dec = nn.Linear(latent_dim, h3 * 4 * 4)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(h3, h2, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(h2, h1, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(h1, 1, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid(),  # matches a Bernoulli-style reconstruction term
        )

    def encode(self, x: torch.Tensor):
        h = self.encoder(x).flatten(1)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps

    def decode(self, z: torch.Tensor):
        h = self.fc_dec(z).view(z.shape[0], -1, 4, 4)
        return self.decoder(h)

    def forward(self, x: torch.Tensor):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return {"sample": recon, "mu": mu, "logvar": logvar, "latent": z}

In [ ]:
def vae_loss(batch_x, output, beta: float = 1.0):
    recon = output["sample"]
    mu = output["mu"]
    logvar = output["logvar"]

    rec = (
        F.binary_cross_entropy(recon, batch_x, reduction="none")
        .flatten(1)
        .sum(1)
        .mean()
    )
    kl = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp()).sum(dim=1).mean()
    loss = rec + beta * kl
    return loss, rec.detach(), kl.detach()


def evaluate_vae(model, loader, beta: float = 1.0):
    model.eval()
    total_loss = 0.0
    total_rec = 0.0
    total_kl = 0.0
    n = 0

    with torch.no_grad():
        for batch in loader:
            x = batch["image"].to(device)
            out = model(x)
            loss, rec, kl = vae_loss(x, out, beta=beta)
            bsz = x.size(0)
            total_loss += loss.item() * bsz
            total_rec += rec.item() * bsz
            total_kl += kl.item() * bsz
            n += bsz

    return {
        "loss": total_loss / n,
        "rec": total_rec / n,
        "kl": total_kl / n,
    }


def train_vae(
    model,
    train_loader,
    val_loader,
    epochs: int = 5,
    lr: float = 2e-3,
    beta: float = 1.0,
):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        tr_loss = tr_rec = tr_kl = 0.0
        n = 0

        for batch in train_loader:
            x = batch["image"].to(device)
            out = model(x)
            loss, rec, kl = vae_loss(x, out, beta=beta)

            opt.zero_grad()
            loss.backward()
            opt.step()

            bsz = x.size(0)
            tr_loss += loss.item() * bsz
            tr_rec += rec.item() * bsz
            tr_kl += kl.item() * bsz
            n += bsz

        train_metrics = {
            "loss": tr_loss / n,
            "rec": tr_rec / n,
            "kl": tr_kl / n,
        }
        val_metrics = evaluate_vae(model, val_loader, beta=beta)
        history.append({"epoch": epoch, "train": train_metrics, "val": val_metrics})

        print(
            f"epoch {epoch:02d} | "
            f"train loss {train_metrics['loss']:.3f} rec {train_metrics['rec']:.3f} kl {train_metrics['kl']:.3f} | "
            f"val loss {val_metrics['loss']:.3f} rec {val_metrics['rec']:.3f} kl {val_metrics['kl']:.3f}"
        )

    return history

In [ ]:
vae = ToyVAE(latent_dim=16, hidden_dims=(32, 64, 128)).to(device)
history = train_vae(vae, train_loader, test_loader, epochs=5, lr=1e-3, beta=1.0)

## 4. Save and reload in Diffusers style

Because the model inherits from `ModelMixin` and `ConfigMixin`, we can standardize serialization with `save_pretrained()` and `from_pretrained()`.


In [ ]:
VAE_DIR = Path("./checkpoints/tiny_vae_mnist")
vae.save_pretrained(VAE_DIR)
vae_reloaded = ToyVAE.from_pretrained(VAE_DIR).to(device)
print("saved to:", VAE_DIR)

In [ ]:
def show_gray_reconstructions(model, loader, n: int = 8):
    model.eval()
    batch = next(iter(loader))
    x = batch["image"][:n].to(device)
    with torch.no_grad():
        recon = model(x)["sample"].cpu()

    plt.figure(figsize=(2 * n, 4))
    for i in range(n):
        plt.subplot(2, n, i + 1)
        plt.imshow(x[i, 0].cpu(), cmap="gray")
        plt.axis("off")
        if i == 0:
            plt.ylabel("input", fontsize=12)

        plt.subplot(2, n, n + i + 1)
        plt.imshow(recon[i, 0], cmap="gray")
        plt.axis("off")
        if i == 0:
            plt.ylabel("recon", fontsize=12)

    plt.tight_layout()
    plt.show()


show_gray_reconstructions(vae_reloaded, test_loader, n=8)

In [ ]:
def sample_from_prior(model, n: int = 16):
    model.eval()
    with torch.no_grad():
        z = torch.randn(n, model.config.latent_dim, device=device)
        x = model.decode(z).cpu()

    rows = int(math.sqrt(n))
    cols = math.ceil(n / rows)
    plt.figure(figsize=(1.8 * cols, 1.8 * rows))
    for i in range(n):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(x[i, 0], cmap="gray")
        plt.axis("off")
    plt.suptitle("Unconditional samples from $p(z)$")
    plt.tight_layout()
    plt.show()


sample_from_prior(vae_reloaded, n=16)

In [ ]:
def latent_interpolation(model, loader, steps: int = 8):
    model.eval()
    batch = next(iter(loader))
    x_a = batch["image"][0:1].to(device)
    x_b = batch["image"][1:2].to(device)

    with torch.no_grad():
        mu_a, _ = model.encode(x_a)
        mu_b, _ = model.encode(x_b)
        alphas = torch.linspace(0, 1, steps, device=device)
        z = torch.stack([(1 - a) * mu_a[0] + a * mu_b[0] for a in alphas], dim=0)
        recon = model.decode(z).cpu()

    plt.figure(figsize=(2 * steps, 2.2))
    for i in range(steps):
        plt.subplot(1, steps, i + 1)
        plt.imshow(recon[i, 0], cmap="gray")
        plt.axis("off")
    plt.suptitle("Latent interpolation")
    plt.tight_layout()
    plt.show()


latent_interpolation(vae_reloaded, test_loader, steps=8)

---

# Part B — Elegant conditional injection with a pretrained T2I-Adapter

## 5. Why do we switch from a VAE to an adapter-based diffusion pipeline?

Part A teaches the latent-variable logic cleanly: encode, regularize, decode.  
Part B asks a different engineering question:

> How do we inject a condition $c$, here, a grayscale line-art image into a large pretrained generator without retraining the whole beast?

The answer used here is **T2I-Adapter**$^{[1]}$. The core idea is pleasantly economical:

- keep the **base diffusion model frozen**;
- learn a small adapter $A_{\psi}$ from the condition image $c$;
- inject the adapter features into the frozen UNet as additional residuals.

In symbols, if $x_0$ is the target colored image and $c$ is the line-art condition, then the diffusion forward process in latent space is

$$
z_0 = \operatorname{VAEEnc}(x_0),
\qquad
z_t = \sqrt{\bar\alpha_t}\,z_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon,
\qquad
\varepsilon \sim \mathcal{N}(0,I).
$$

The adapter produces multi-scale control features

$$
r = A_{\psi}(c) = \{r^{(1)}, r^{(2)}, \dots, r^{(L)}\},
$$

and the frozen UNet predicts noise with conditional residual injection:

$$
\hat\varepsilon
=
\varepsilon_{\theta}\!\bigl(
z_t,\ t,\ h_{\text{text}};\ r
\bigr).
$$

Here $h_{\text{text}}$ denotes the text embedding. In our teaching setup, we deliberately use the **empty prompt** during training, so the line drawing carries the semantics while the text encoder remains politely silent.

The training objective in the provided script is the standard diffusion MSE loss:

$$
\mathcal{L}_{\text{adapter}}
=
\mathbb{E}_{x_0,c,\varepsilon,t}
\left[
\left\|
\hat\varepsilon - \varepsilon
\right\|_2^2
\right].
$$

Because $\theta$ is frozen and only $\psi$ is updated, the method is efficient in parameters, though not always in wall-clock time. GPUs, like graduate students, also need boundaries.

For this course, **training has already been completed**. We therefore focus only on **loading and using** the adapter.

[1] :Mou, Chong, et al. "T2i-adapter: Learning adapters to dig out more controllable ability for text-to-image diffusion models." Proceedings of the AAAI conference on artificial intelligence. Vol. 38. No. 5. 2024.


In [ ]:
from torchvision.transforms import transforms
from diffusers import StableDiffusionAdapterPipeline, T2IAdapter


## 6. Dataset interface (aligned with the Lab 4 line-art colorization setup)

We follow the same data convention as the training script:

- `gray_image`: the control input, i.e. the grayscale line drawing;
- `target_image`: the supervision target, i.e. the colored image;
- `id`: sample identifier.

The training script reads `train.parquet` and `test.parquet` via `datasets.load_dataset(...)`, applies a **normalized transform** for the target image and a **non-normalized transform** for the control image, and packs them into a collate function. We mirror that logic below for inference so that the notebook matches the trained adapter interface.


In [ ]:
DATA_DIR = Path("./data")
RESOLUTION = 512

conditioning_transforms = transforms.Compose(
    [
        transforms.Resize(RESOLUTION, interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.CenterCrop(RESOLUTION),
        transforms.ToTensor(),  # kept in [0, 1] for adapter conditioning
    ]
)

target_vis_transforms = transforms.Compose(
    [
        transforms.Resize(RESOLUTION, interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.CenterCrop(RESOLUTION),
        transforms.ToTensor(),  # visualization only
    ]
)

dataset_color = load_dataset(
    "parquet",
    data_files={
        "train": str(DATA_DIR / "train.parquet"),
        "test": str(DATA_DIR / "test.parquet"),
    },
)

dataset_color


In [ ]:
def make_adapter_batch(split="test", n=4):
    ds = dataset_color[split].select(range(min(n, len(dataset_color[split]))))
    batch = []
    for sample in ds:
        cond = conditioning_transforms(sample["gray_image"].convert("RGB"))
        tgt = None
        if "target_image" in sample and sample["target_image"] is not None:
            tgt = target_vis_transforms(sample["target_image"].convert("RGB"))
        batch.append(
            {
                "id": sample["id"],
                "src": cond,
                "dst": tgt,
            }
        )
    return batch

def show_condition_target_pairs(batch):
    n = len(batch)
    plt.figure(figsize=(4 * n, 6))
    for i, item in enumerate(batch):
        plt.subplot(2, n, i + 1)
        plt.imshow(item["src"].permute(1, 2, 0).cpu().numpy())
        plt.axis("off")
        plt.title(f"condition #{item['id']}")

        plt.subplot(2, n, n + i + 1)
        if item["dst"] is not None:
            plt.imshow(item["dst"].permute(1, 2, 0).cpu().numpy())
            plt.title("target")
        else:
            plt.text(0.5, 0.5, "target unavailable", ha="center", va="center")
            plt.title("target")
        plt.axis("off")
    plt.tight_layout()

adapter_batch = make_adapter_batch(split="test", n=4)
show_condition_target_pairs(adapter_batch)


## 7. Load the pretrained adapter

The training script builds the adapter as

$$
A_{\psi}:\mathbb{R}^{H\times W\times 3}\to \{r^{(l)}\}_{l=1}^{L},
$$

with channel widths matched to the frozen UNet blocks. It then saves the learned weights through `save_pretrained(...)`, so we load them in exactly the same style.

Please set:

- `BASE_MODEL_PATH` to the same Stable Diffusion checkpoint used during training;
- `ADAPTER_PATH` to the directory containing the pretrained adapter weights prepared by the course staff.


In [ ]:
BASE_MODEL_PATH = "stable-diffusion-v1-5/stable-diffusion-v1-5"   
ADAPTER_PATH = "./checkpoints/color-adapter/checkpoint-500/t2i_adapter"

adapter = T2IAdapter.from_pretrained(ADAPTER_PATH, torch_dtype=torch.float16 if device.type == "cuda" else torch.float32)

pipe = StableDiffusionAdapterPipeline.from_pretrained(
    BASE_MODEL_PATH,
    adapter=adapter,
    safety_checker=None,
    requires_safety_checker=False,
    torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
)

pipe = pipe.to(device)
pipe.set_progress_bar_config(disable=True)

print("Pipeline loaded.")


## 8. Run controlled generation

Because the adapter was trained with an empty prompt in the reference script, the most faithful classroom inference setting is also

$$
h_{\text{text}} = \operatorname{TextEnc}(\texttt{""}).
$$

At inference time, the adapter extracts control features from the line art, and the pipeline denoises latents conditioned on those features. A useful knob is `adapter_conditioning_scale`:

$$
r \leftarrow s\,r,
$$

where \(s>0\) controls how strongly the condition enters the frozen UNet.  
Roughly speaking:

- smaller \(s\): weaker control, more freedom, more “creative misunderstanding”;
- larger \(s\): stronger structural faithfulness, less improvisation.

For line-art colorization, values around `0.8` to `1.5` are usually a sensible starting point.


In [ ]:
from torchvision.transforms.functional import to_pil_image

def run_adapter_inference(
    batch,
    prompt="",
    negative_prompt="",
    num_inference_steps=30,
    guidance_scale=7.5,
    adapter_conditioning_scale=1.0,
    seed=42,
):
    cond_images = [to_pil_image(item["src"]) for item in batch]
    generator = torch.Generator(device=device).manual_seed(seed)

    outputs = pipe(
        prompt=[prompt] * len(batch),
        negative_prompt=[negative_prompt] * len(batch),
        image=cond_images,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        adapter_conditioning_scale=adapter_conditioning_scale,
        generator=generator,
        height=RESOLUTION,
        width=RESOLUTION,
    ).images
    return outputs

generated = run_adapter_inference(
    adapter_batch,
    prompt="",
    negative_prompt="",
    num_inference_steps=30,
    guidance_scale=7.5,
    adapter_conditioning_scale=1.0,
    seed=42,
)


In [ ]:
def show_colorization_results(batch, generated):
    n = len(batch)
    has_target = all(item["dst"] is not None for item in batch)
    rows = 3 if has_target else 2

    plt.figure(figsize=(4 * n, 4 * rows))
    for i, (item, pred) in enumerate(zip(batch, generated)):
        plt.subplot(rows, n, i + 1)
        plt.imshow(item["src"].permute(1, 2, 0).cpu().numpy())
        plt.axis("off")
        plt.title(f"condition #{item['id']}")

        plt.subplot(rows, n, n + i + 1)
        plt.imshow(pred)
        plt.axis("off")
        plt.title("generated")

        if has_target:
            plt.subplot(rows, n, 2 * n + i + 1)
            plt.imshow(item["dst"].permute(1, 2, 0).cpu().numpy())
            plt.axis("off")
            plt.title("target")

    plt.tight_layout()

show_colorization_results(adapter_batch, generated)


## 9. A minimal ablation: what does control strength do?

We vary only the control scale $s$, while keeping the random seed fixed. This isolates the effect of conditional injection rather than mixing it with sampling randomness.


In [ ]:
scales = [0.5, 1.0, 1.5]
example = adapter_batch[:1]

results_by_scale = {}
for s in scales:
    results_by_scale[s] = run_adapter_inference(
        example,
        prompt="",
        negative_prompt="",
        num_inference_steps=30,
        guidance_scale=7.5,
        adapter_conditioning_scale=s,
        seed=42,
    )[0]

plt.figure(figsize=(4 * (len(scales) + 1), 4))
plt.subplot(1, len(scales) + 1, 1)
plt.imshow(example[0]["src"].permute(1, 2, 0).cpu().numpy())
plt.axis("off")
plt.title("condition")

for j, s in enumerate(scales, start=2):
    plt.subplot(1, len(scales) + 1, j)
    plt.imshow(results_by_scale[s])
    plt.axis("off")
    plt.title(f"scale = {s}")

plt.tight_layout()


## 10. What should you guys remember?

1. **VAE logic** gives the latent viewpoint: encode $x$ to $z$, regularize toward a simple prior, and decode back.
2. **Latent diffusion** moves generation into the VAE latent space, where sampling is cheaper and more structured.
3. **T2I-Adapter** adds condition control by learning a small module $A_{\psi}(c)$ while freezing the expensive backbone.
4. The actual conditional mechanism is simple:

$$
c \xrightarrow{A_{\psi}} r
\quad\Longrightarrow\quad
\hat\varepsilon
=
\varepsilon_{\theta}(z_t,t,h_{\text{text}};r).
$$

5. In this lab, the adapter is the only learned control module; the rest of the pipeline is reused. This is good engineering and also emotionally healthier than fine-tuning everything on a Tuesday night.
